# Trích landmark bàn tay từ HaGRID — chạy trên Kaggle

**Dự án:** AI Hand-Gesture Authenticated Smart Door Lock
**Notebook này chạy trên Kaggle**, không chạy được ở máy cá nhân.

---

## Mục đích — đọc kỹ, nó quyết định cách diễn giải kết quả

Dữ liệu HaGRID dùng làm **tập kiểm thử độc lập**, **KHÔNG** trộn vào tập huấn luyện
theo mặc định.

Mô hình được huấn luyện trên dữ liệu tự thu bằng webcam của chính người dùng.
HaGRID trả lời một câu hỏi khác: *"mô hình có nhận được bàn tay của người khác,
trong điều kiện chụp khác, hay không?"*

**Con số trên HaGRID sẽ thấp hơn tập validation tự thu. Đó là kết quả mong đợi và
trung thực, không phải lỗi cần sửa.** Khoảng cách giữa hai con số chính là thứ đáng
báo cáo — nó đo mức độ mô hình phụ thuộc vào người, nền và ánh sáng lúc thu dữ liệu.

## Vì sao phải chạy trên Kaggle

HaGRID nặng hàng trăm GB. Kaggle host sẵn dataset nên notebook đọc thẳng từ
`/kaggle/input`, không phải tải về máy. Thứ duy nhất quay về máy người dùng là vài
file CSV cỡ vài MB.

## Đầu ra

| File | Nội dung |
|---|---|
| `keypoint_hagrid.csv` | **đúng schema** với `vision/model/keypoint.csv`: cột 1 `class_id`, 42 cột toạ độ |
| `keypoint_hagrid_meta.csv` | metadata để truy vết: `image_id`, `hagrid_label`, `class_id`, `mp_handedness`, `leading_hand`, `bbox_area_frac` |
| `extract_report.json` | số ảnh thử/thành công/thất bại theo lớp, tỉ lệ khớp handedness, tham số đã dùng |

Tải cả ba về `vision/model/` rồi chạy `vision/eval_external.py`.

---

## Cell 1 — Khám phá dataset, không giả định gì

Cell này **không trích xuất gì cả**. Nó chỉ đi tìm và in ra sự thật:

1. Dataset HaGRID nằm ở đường dẫn nào trong `/kaggle/input` (tên thư mục **không**
   viết cứng, vì mỗi bản HaGRID trên Kaggle đặt tên một kiểu).
2. Danh sách **tên lớp thật** trong file annotation.
3. Cấu trúc một bản ghi annotation mẫu, in nguyên văn.
4. **Định dạng bounding box**: toạ độ tuyệt đối hay chuẩn hoá, thứ tự `x,y,w,h` hay
   `x1,y1,x2,y2`.

**Chạy cell này trước, đọc kỹ output, rồi mới chạy tiếp.** Nếu tên lớp in ra không
khớp bảng ánh xạ ở Cell 4 thì dừng lại — đừng tự đoán ánh xạ thay thế.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("=" * 74)
print("1. NOI DUNG /kaggle/input")
print("=" * 74)
if not INPUT_ROOT.exists():
    raise SystemExit("Khong thay /kaggle/input. Notebook nay chi chay duoc tren Kaggle.")

for d in sorted(INPUT_ROOT.iterdir()):
    print("  ", d.name)

# --- Tim file annotation JSON, khong viet cung ten thu muc ---
print()
print("=" * 74)
print("2. TIM FILE ANNOTATION (.json)")
print("=" * 74)
ann_files = []
for root, dirs, files in os.walk(INPUT_ROOT):
    if root.count(os.sep) - str(INPUT_ROOT).count(os.sep) > 4:
        dirs[:] = []            # khong di qua sau, tranh cham
        continue
    for f in files:
        if f.endswith(".json"):
            ann_files.append(Path(root) / f)
    if len(ann_files) > 200:
        break

print(f"   Tim thay {len(ann_files)} file .json. 20 file dau:")
for p in ann_files[:20]:
    print(f"      {p.relative_to(INPUT_ROOT)}  ({p.stat().st_size/1024:.0f} KB)")

if not ann_files:
    raise SystemExit("Khong tim thay file annotation nao. Kiem tra lai da Add Data chua.")

# --- Ten lop that: thuong la ten file annotation (vd fist.json) ---
print()
print("=" * 74)
print("3. TEN LOP THAT (theo ten file annotation)")
print("=" * 74)
ten_lop = sorted({p.stem for p in ann_files})
print(f"   {len(ten_lop)} lop:")
for t in ten_lop:
    print("      ", t)

# --- Doc thu 1 file, in nguyen van 1 ban ghi ---
print()
print("=" * 74)
print("4. MOT BAN GHI ANNOTATION, IN NGUYEN VAN")
print("=" * 74)
mau_path = ann_files[0]
with open(mau_path, "r") as f:
    mau = json.load(f)
print("   File:", mau_path.relative_to(INPUT_ROOT))
print("   Kieu du lieu goc:", type(mau).__name__, "| so phan tu:", len(mau))

khoa_dau = list(mau.keys())[0] if isinstance(mau, dict) else 0
ban_ghi = mau[khoa_dau]
print("   image_id mau:", khoa_dau)
print("   Cac truong  :", list(ban_ghi.keys()) if isinstance(ban_ghi, dict) else "(khong phai dict)")
print()
print(json.dumps(ban_ghi, indent=4)[:1200])

# --- Xac dinh dinh dang bbox, KHONG gia dinh ---
print()
print("=" * 74)
print("5. DINH DANG BOUNDING BOX")
print("=" * 74)
bb = ban_ghi.get("bboxes", [])
print("   Gia tri bboxes nguyen van:", bb)
if bb:
    v = bb[0]
    print("   So phan tu moi bbox:", len(v))
    trong_01 = all(0.0 <= x <= 1.0 for x in v)
    print("   Moi gia tri nam trong [0,1]:", trong_01,
          "->", "CHUAN HOA" if trong_01 else "TUYET DOI (pixel)")
    # Neu la x1,y1,x2,y2 thi v[2]>v[0] va v[3]>v[1]; neu la x,y,w,h thi v[2],v[3] la kich thuoc
    kieu_xywh = (v[2] + v[0] <= 1.05) and (v[3] + v[1] <= 1.05) if trong_01 else None
    print(f"   x+w={v[0]+v[2]:.3f}  y+h={v[1]+v[3]:.3f}   (neu <= 1.0 thi la dang x,y,w,h)")
    print("   -> DOAN:", "x,y,w,h" if kieu_xywh else "x1,y1,x2,y2 (hoac tuyet doi)")
    print()
    print("   *** DOC KY DONG TREN. Cell 4 mac dinh hieu la CHUAN HOA dang x,y,w,h. ***")
    print("   *** Neu khac, sua bien BBOX_FORMAT o dau Cell 4 truoc khi chay.        ***")

# --- Dem so anh moi lop ---
print()
print("=" * 74)
print("6. SO BAN GHI MOI LOP")
print("=" * 74)
dem = {}
for p in ann_files:
    try:
        with open(p, "r") as f:
            dem[p.stem] = len(json.load(f))
    except Exception as e:
        dem[p.stem] = f"loi doc: {type(e).__name__}"
for k in sorted(dem):
    print(f"   {k:<22}{dem[k]}")

# --- Tim thu muc anh ---
print()
print("=" * 74)
print("7. THU MUC ANH")
print("=" * 74)
duoi_anh = {".jpg", ".jpeg", ".png"}
thu_muc_anh = Counter()
for root, dirs, files in os.walk(INPUT_ROOT):
    n = sum(1 for f in files if Path(f).suffix.lower() in duoi_anh)
    if n:
        thu_muc_anh[root] = n
    if len(thu_muc_anh) > 60:
        break
for d, n in list(thu_muc_anh.items())[:30]:
    print(f"   {n:>7} anh  {Path(d).relative_to(INPUT_ROOT)}")

print()
print("=" * 74)
print("XONG CELL 1. Doc ky muc 3 (ten lop) va muc 5 (dinh dang bbox) truoc khi chay tiep.")
print("=" * 74)

---

## Cell 2 — Cài đặt MediaPipe và tải model

> **⚠️ Phải bật Internet cho notebook:** Kaggle → panel bên phải → **Settings** →
> **Internet: On**. Không bật thì `pip install` và tải model đều hỏng.
>
> Tài khoản Kaggle phải **xác minh số điện thoại** mới bật được Internet. Nếu mục
> Internet bị khoá, vào Settings tài khoản xác minh trước.

Phiên bản MediaPipe được ghi lại vào báo cáo — nếu bản trên Kaggle khác bản ở máy
(`0.10.31`) thì kết quả có thể lệch chút ít, và cần nêu trong báo cáo.

In [ ]:
import subprocess
import sys
import urllib.request
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe"], check=True)

import mediapipe as mp
print("mediapipe:", mp.__version__)
print("(o may local dang dung 0.10.31 — khac ban thi ghi ro trong bao cao)")

MODEL_PATH = Path("/kaggle/working/hand_landmarker.task")
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/1/hand_landmarker.task")

if not MODEL_PATH.exists():
    print("Dang tai hand_landmarker.task ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f"model: {MODEL_PATH} ({MODEL_PATH.stat().st_size/1024:.0f} KB)")
print("nguon:", MODEL_URL)

---

## Cell 3 — Tiền xử lý landmark

Khối dưới đây là **bản chép nguyên văn** từ `vision/common.py`. Notebook Kaggle
không import được file của dự án nên buộc phải chép.

Chép thì có nguy cơ hai bản lệch nhau theo thời gian, nên ở phía máy local có
`tests/test_preprocess_parity.py` làm việc sau: trích đúng khối nằm giữa hai dòng
đánh dấu `BEGIN`/`END` ra khỏi notebook này, chạy nó, rồi so từng con số với
`vision/common.preprocess_landmarks`. **Sai số tuyệt đối tối đa phải bằng 0.**

**Đừng sửa khối này trực tiếp.** Muốn đổi thì sửa `vision/common.py` rồi chép lại,
sau đó chạy test parity để xác nhận.

> Lệch một bước tiền xử lý là toàn bộ dữ liệu HaGRID thành nhiễu, và con số kiểm thử
> sẽ vô nghĩa mà không có dấu hiệu gì báo ra ngoài.

In [ ]:
# === BEGIN preprocess_landmarks (COPY NGUYEN VAN tu vision/common.py) ===
import numpy as np

NUM_LANDMARKS = 21
FEATURE_DIM = NUM_LANDMARKS * 2  # 42


def _to_xy_array(hand_landmarks) -> np.ndarray:
    """Gom 21 diem moc thanh mang (21, 2).

    Chap nhan 2 dang dau vao:
      - list cac object co thuoc tinh .x .y  (dang MediaPipe Tasks tra ve)
      - list cac cap (x, y) hoac (x, y, z)   (dung cho unit test, khong can
        phai dung toi MediaPipe)
    """
    pts = []
    for lm in hand_landmarks:
        if hasattr(lm, "x") and hasattr(lm, "y"):
            pts.append((float(lm.x), float(lm.y)))
        else:
            pts.append((float(lm[0]), float(lm[1])))

    arr = np.asarray(pts, dtype=np.float32)
    if arr.shape != (NUM_LANDMARKS, 2):
        raise ValueError(
            f"Can dung {NUM_LANDMARKS} diem moc, nhan duoc {arr.shape[0]}"
        )
    return arr


def preprocess_landmarks(hand_landmarks, handedness: str) -> np.ndarray:
    """Bien 21 diem moc ban tay thanh vector dac trung 42 chieu.

    Ba buoc chuan hoa, lam dung thu tu nay:

      1. DOI GOC ve co tay (diem 0). Nho vay tay o goc trai hay goc phai
         khung hinh deu cho ra cung mot vector — bo di anh huong vi tri.

      2. LAT DAU TRUC X neu la tay trai. Moi thu duoc quy ve "tay phai", nen
         model chi phai hoc mot nua so truong hop, va nguoi dung tay trai hay
         tay phai deu dung duoc.

      3. CHIA CHO GIA TRI TUYET DOI LON NHAT (tinh chung ca x lan y). Nho vay
         tay gan camera (to) hay xa camera (nho) deu cho ra cung mot vector —
         bo di anh huong khoang cach.

    Tham so:
      hand_landmarks : 21 diem moc, toa do chuan hoa 0..1 cua MediaPipe
      handedness     : "Left" hoac "Right" (result.handedness[0][0].category_name)

    Tra ve:
      np.ndarray shape (42,) dtype float32, duoi phang theo thu tu
      x0, y0, x1, y1, ..., x20, y20
    """
    arr = _to_xy_array(hand_landmarks)

    # Buoc 1 — doi goc toa do ve co tay.
    arr = arr - arr[0]

    # Buoc 2 — quy tay trai ve tay phai bang cach lat dau truc x.
    if handedness == "Left":
        arr[:, 0] = -arr[:, 0]

    # Buoc 3 — chuan hoa ty le. Lay max tren CA x va y de giu dung ty le
    # ngang/doc cua ban tay (neu chia rieng tung truc se lam meo hinh dang).
    max_abs = float(np.max(np.abs(arr)))
    if max_abs > 0.0:
        arr = arr / max_abs

    return arr.flatten().astype(np.float32)
# === END preprocess_landmarks ===

---

## Cell 4 — Trích xuất landmark

### Bốn quyết định kỹ thuật, kèm lý do

**1. `RunningMode.IMAGE`, không phải `VIDEO`.**
Ảnh HaGRID là các ảnh rời rạc, không phải chuỗi khung liên tiếp. Ở chế độ `VIDEO`,
MediaPipe giữ trạng thái tracker giữa các lần gọi và cho rằng khung sau nối tiếp
khung trước — với ảnh rời rạc điều đó làm kết quả sai lệch khó lường.

**2. Crop theo bounding box rồi nới rộng 20% mỗi cạnh.**
Ảnh HaGRID chụp người đứng xa tới 4 mét, bàn tay chiếm rất ít diện tích khung hình.
Đưa cả ảnh toàn cảnh vào thì MediaPipe bỏ sót phần lớn. Nới rộng vì bbox sát quá sẽ
cắt mất đầu ngón tay, mà đầu ngón chính là phần quan trọng nhất để phân biệt cử chỉ.

**3. Lật ngang ảnh crop trước khi đưa vào MediaPipe.**
Đây là quy ước **giống hệt** pipeline local: cả `collect_data.py` (dòng 193) lẫn
`gesture_lock.py` (dòng 505) đều gọi `cv2.flip(frame, 1)` ngay trước MediaPipe.
Chi tiết vì sao quan trọng: xem ô markdown ngay dưới.

**4. Đếm mọi thứ, không lặng lẽ bỏ qua.**
Mỗi lớp ghi lại: số ảnh đã thử, số ảnh MediaPipe không tìm thấy tay, số ảnh lỗi đọc
file. Một lớp có tỉ lệ phát hiện thấp mà không ai biết sẽ làm lệch toàn bộ kết luận.

### Vì sao phải lật ảnh — và vì sao nó *không* ảnh hưởng theo cách bạn tưởng

MediaPipe xác định nhãn `handedness` với **giả định ảnh đầu vào là ảnh gương**
(selfie). Ảnh HaGRID là ảnh chụp thường, **không** phải ảnh gương.

Hệ quả cụ thể:

| Notebook làm gì | MediaPipe báo handedness | Khớp với `leading_hand` của HaGRID |
|---|---|---|
| **có lật** (đúng quy ước local) | đúng tay thật | cao — kỳ vọng > 80% |
| không lật | **ngược** tay thật | gần 0% |

Nhưng — và đây là điểm dễ hiểu nhầm — **vector đặc trưng vẫn giống nhau ở cả hai
cách**. Lý do: `preprocess_landmarks` đã quy mọi bàn tay về "tay phải" bằng bước lật
trục x. Khi ảnh bị lật, nhãn handedness cũng lật theo, và hai phép lật triệt tiêu
nhau. Điều này đã đo được ở máy local: trên 500 bàn tay ngẫu nhiên, sai lệch tối đa
giữa hai đường là `2.98e-07`, tức đúng bằng mức làm tròn của `float32`.

**Vậy rủi ro thật nằm ở đâu?** Ở chỗ MediaPipe báo **sai** handedness. Nếu nó báo sai,
bước lật áp nhầm và vector lệch tới `2.0` trên thang `[-1, 1]` — tức hỏng hoàn toàn.
Trên ảnh HaGRID (tay nhỏ, xa, độ phân giải thấp) khả năng báo sai cao hơn hẳn so với
bàn tay cận cảnh trên webcam.

**Nên tỉ lệ khớp `leading_hand` không phải để kiểm tra quy ước lật — nó là thước đo
mức độ tin được của nhãn handedness trên HaGRID.** Tỉ lệ thấp nghĩa là một phần dữ
liệu đã bị lật nhầm và trở thành nhiễu. Đó là lý do phải dừng lại kiểm tra khi tỉ lệ
này dưới 80%.

**Không dùng `leading_hand` thay cho handedness của MediaPipe** — pipeline local dùng
handedness của MediaPipe, nên dữ liệu HaGRID cũng phải dùng đúng nguồn đó thì mới so
sánh được. `leading_hand` chỉ dùng để đối chiếu và báo cáo.

In [ ]:
# ===================== THAM SO — CHINH O DAY =====================
MAX_PER_CLASS = 300          # so mau toi da moi lop
BBOX_EXPAND   = 0.20         # noi rong 20% moi canh (be ngang thanh 1.4 lan)
BBOX_FORMAT   = "xywh_norm"  # doi neu Cell 1 bao khac: "xywh_abs" | "xyxy_norm" | "xyxy_abs"
FLIP_TRUOC_MEDIAPIPE = True  # GIU NGUYEN — dung quy uoc voi collect_data.py / gesture_lock.py
CHECKPOINT_MOI = 500         # ghi CSV tam moi bao nhieu mau
# =================================================================

# Anh xa lop HaGRID -> class_id cua du an (muc B3)
HAGRID_TO_LOCAL = {
    "fist":       (0, "fist"),
    "palm":       (1, "palm"),
    "peace":      (2, "victory"),
    "like":       (3, "thumb_up"),
    "one":        (4, "point_up"),
    "ok":         (5, "ok"),
    "no_gesture": (6, "none"),
}
# Co y LOAI BO, khong anh xa:
#   peace_inverted, stop_inverted, two_up_inverted : tay up nguoc, hinh dang khac han
#   stop                                           : long ban tay khep ngon, de lan voi palm
#   moi lop con lai khong co trong bang

import csv
import json
import os
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

WORK = Path("/kaggle/working")

# ---------------------------------------------------------------- kiem tra lop
ten_lop_that = sorted({p.stem for p in ann_files})
thieu = [k for k in HAGRID_TO_LOCAL if k not in ten_lop_that]
if thieu:
    print("TEN LOP THAT TRONG DATASET:")
    for t in ten_lop_that:
        print("   ", t)
    raise SystemExit(
        f"\nDUNG LAI. Thieu lop {thieu} trong dataset nay.\n"
        "Khong tu doan anh xa thay the. Doi HAGRID_TO_LOCAL cho khop ten lop that,\n"
        "hoac dung ban HaGRID khac."
    )
print("Da xac nhan: ca 7 lop can dung deu co trong dataset.")

# ------------------------------------------------------------- tim duong dan anh
def tim_anh(image_id: str, label: str):
    """Tim file anh theo image_id. Thu vai kieu to chuc thu muc thuong gap."""
    for thu_muc in THU_MUC_ANH_UNG_VIEN:
        for duoi in (".jpg", ".jpeg", ".png", ".JPG"):
            p = Path(thu_muc) / label / (image_id + duoi)
            if p.exists():
                return p
            p = Path(thu_muc) / (image_id + duoi)
            if p.exists():
                return p
    return None


# Cac thu muc co anh, lay tu ket qua Cell 1
THU_MUC_ANH_UNG_VIEN = list(thu_muc_anh.keys())
print(f"So thu muc anh ung vien: {len(THU_MUC_ANH_UNG_VIEN)}")

# ------------------------------------------------------------------ doi bbox
def bbox_sang_pixel(bb, W, H):
    """Doi bbox ve (x1, y1, x2, y2) tinh bang pixel, theo BBOX_FORMAT."""
    a, b, c, d = bb[:4]
    if BBOX_FORMAT == "xywh_norm":
        x1, y1, x2, y2 = a * W, b * H, (a + c) * W, (b + d) * H
    elif BBOX_FORMAT == "xywh_abs":
        x1, y1, x2, y2 = a, b, a + c, b + d
    elif BBOX_FORMAT == "xyxy_norm":
        x1, y1, x2, y2 = a * W, b * H, c * W, d * H
    elif BBOX_FORMAT == "xyxy_abs":
        x1, y1, x2, y2 = a, b, c, d
    else:
        raise ValueError(f"BBOX_FORMAT khong hop le: {BBOX_FORMAT}")
    return x1, y1, x2, y2


def crop_noi_rong(img, bb):
    """Crop vung tay, noi rong BBOX_EXPAND moi canh. Tra ve (crop, ty_le_dien_tich)."""
    H, W = img.shape[:2]
    x1, y1, x2, y2 = bbox_sang_pixel(bb, W, H)
    w, h = x2 - x1, y2 - y1
    if w <= 1 or h <= 1:
        return None, 0.0

    dx, dy = w * BBOX_EXPAND, h * BBOX_EXPAND
    X1 = max(0, int(x1 - dx)); Y1 = max(0, int(y1 - dy))
    X2 = min(W, int(x2 + dx)); Y2 = min(H, int(y2 + dy))
    if X2 - X1 < 8 or Y2 - Y1 < 8:
        return None, 0.0
    return img[Y1:Y2, X1:X2], (w * h) / float(W * H)


# ------------------------------------------------------------ HandLandmarker
options = mp_vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=mp_vision.RunningMode.IMAGE,   # IMAGE, khong phai VIDEO
    num_hands=1,
)

rows = []       # [class_id, 42 gia tri]
meta = []       # metadata truy vet
thong_ke = defaultdict(lambda: {"da_thu": 0, "thanh_cong": 0,
                                "khong_thay_tay": 0, "loi_doc_file": 0,
                                "bbox_hong": 0})
khop_handed = {"khop": 0, "lech": 0, "thieu_nhan": 0}


def ghi_checkpoint():
    with open(WORK / "keypoint_hagrid.csv", "w", newline="") as f:
        csv.writer(f).writerows(rows)


t_bat_dau = time.time()
with mp_vision.HandLandmarker.create_from_options(options) as landmarker:
    for hagrid_label, (class_id, ten_local) in HAGRID_TO_LOCAL.items():
        ann_path = next(p for p in ann_files if p.stem == hagrid_label)
        with open(ann_path, "r") as f:
            ann = json.load(f)

        tk = thong_ke[hagrid_label]
        print(f"\n[{hagrid_label:<12} -> {class_id} {ten_local:<9}] {len(ann)} ban ghi, "
              f"lay toi da {MAX_PER_CLASS}")

        for image_id, ban_ghi in ann.items():
            if tk["thanh_cong"] >= MAX_PER_CLASS:
                break
            tk["da_thu"] += 1

            img_path = tim_anh(image_id, hagrid_label)
            if img_path is None:
                tk["loi_doc_file"] += 1
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                tk["loi_doc_file"] += 1
                continue

            bboxes = ban_ghi.get("bboxes") or []
            if not bboxes:
                tk["bbox_hong"] += 1
                continue

            crop, dien_tich = crop_noi_rong(img, bboxes[0])
            if crop is None:
                tk["bbox_hong"] += 1
                continue

            # Quy uoc lat GIONG HET pipeline local
            if FLIP_TRUOC_MEDIAPIPE:
                crop = cv2.flip(crop, 1)

            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            kq = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if not kq.hand_landmarks:
                tk["khong_thay_tay"] += 1
                continue

            lms = kq.hand_landmarks[0]
            mp_handed = kq.handedness[0][0].category_name
            dac_trung = preprocess_landmarks(lms, mp_handed)

            rows.append([class_id] + [f"{v:.6f}" for v in dac_trung])

            leading = ban_ghi.get("leading_hand")
            if leading is None:
                khop_handed["thieu_nhan"] += 1
            elif str(leading).lower() == mp_handed.lower():
                khop_handed["khop"] += 1
            else:
                khop_handed["lech"] += 1

            meta.append({
                "image_id": image_id,
                "hagrid_label": hagrid_label,
                "class_id": class_id,
                "mp_handedness": mp_handed,
                "leading_hand": leading,
                "bbox_area_frac": round(dien_tich, 6),
            })
            tk["thanh_cong"] += 1

            if len(rows) % CHECKPOINT_MOI == 0:
                ghi_checkpoint()
                print(f"    ... checkpoint {len(rows)} mau "
                      f"({time.time()-t_bat_dau:.0f}s)")

        print(f"    da thu {tk['da_thu']:>5} | thanh cong {tk['thanh_cong']:>4} "
              f"| khong thay tay {tk['khong_thay_tay']:>4} "
              f"| loi doc file {tk['loi_doc_file']:>4} | bbox hong {tk['bbox_hong']:>3}")

ghi_checkpoint()
print(f"\nXONG. {len(rows)} mau trong {time.time()-t_bat_dau:.0f}s")

---

## Cell 5 — Xuất kết quả

`keypoint_hagrid.csv` giữ **đúng schema** với `vision/model/keypoint.csv`: cột 1 là
`class_id`, 42 cột toạ độ, **không thêm cột nào**. Thêm cột sẽ làm hỏng mọi đoạn code
đang đọc dữ liệu (notebook train và `eval_external.py` đều `np.loadtxt` theo số cột).

Metadata để riêng ở `keypoint_hagrid_meta.csv`, cùng thứ tự dòng với file trên.

In [ ]:
import json

# ---------------------------------------------------- 1. keypoint_hagrid.csv
ghi_checkpoint()
n_cot = len(rows[0]) if rows else 0
print(f"keypoint_hagrid.csv       : {len(rows)} dong x {n_cot} cot "
      f"({'DUNG schema' if n_cot == 43 else '*** SAI schema, phai la 43 ***'})")

# ------------------------------------------------ 2. keypoint_hagrid_meta.csv
COT_META = ["image_id", "hagrid_label", "class_id",
            "mp_handedness", "leading_hand", "bbox_area_frac"]
with open(WORK / "keypoint_hagrid_meta.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COT_META)
    w.writeheader()
    w.writerows(meta)
print(f"keypoint_hagrid_meta.csv  : {len(meta)} dong x {len(COT_META)} cot")

# --------------------------------------------------- 3. ty le khop handedness
tong_khop = khop_handed["khop"] + khop_handed["lech"]
ty_le_khop = khop_handed["khop"] / tong_khop if tong_khop else None

# ----------------------------------------------------- 4. extract_report.json
report = {
    "tham_so": {
        "MAX_PER_CLASS": MAX_PER_CLASS,
        "BBOX_EXPAND": BBOX_EXPAND,
        "BBOX_FORMAT": BBOX_FORMAT,
        "FLIP_TRUOC_MEDIAPIPE": FLIP_TRUOC_MEDIAPIPE,
        "quy_uoc_lat": ("cv2.flip(crop, 1) ngay truoc MediaPipe — giong het "
                        "collect_data.py va gesture_lock.py"),
        "running_mode": "IMAGE",
        "num_hands": 1,
        "mediapipe_version": mp.__version__,
    },
    "anh_xa_lop": {k: {"class_id": v[0], "ten_local": v[1]}
                   for k, v in HAGRID_TO_LOCAL.items()},
    "lop_bi_loai": ["peace_inverted", "stop_inverted", "two_up_inverted", "stop"],
    "theo_lop": {k: dict(v) for k, v in thong_ke.items()},
    "handedness": {
        **khop_handed,
        "ty_le_khop": ty_le_khop,
        "ghi_chu": ("So sanh handedness cua MediaPipe voi truong leading_hand cua "
                    "HaGRID. Duoi 0.80 la dau hieu nhan handedness khong tin duoc "
                    "tren anh nay — dung lai kiem tra truoc khi dung so lieu."),
    },
    "tong_so_mau": len(rows),
}
with open(WORK / "extract_report.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print("extract_report.json       : da ghi")

# ------------------------------------------------------- 5. Bang tom tat
print()
print("=" * 78)
print("TY LE PHAT HIEN THEO LOP")
print("=" * 78)
print(f"{'HaGRID':<14}{'->':<4}{'local':<10}{'da thu':>8}{'thanh cong':>12}"
      f"{'ty le':>9}{'':>4}")
print("-" * 78)
canh_bao = []
for k, (cid, ten) in HAGRID_TO_LOCAL.items():
    tk = thong_ke[k]
    ty_le = tk["thanh_cong"] / tk["da_thu"] if tk["da_thu"] else 0.0
    dau = ""
    if ty_le < 0.50:
        dau = "  <-- DUOI 50%"
        canh_bao.append((k, ty_le))
    print(f"{k:<14}{'->':<4}{ten:<10}{tk['da_thu']:>8}{tk['thanh_cong']:>12}"
          f"{ty_le:>8.1%}{dau}")
print("-" * 78)
print(f"{'TONG':<28}{sum(t['da_thu'] for t in thong_ke.values()):>8}"
      f"{len(rows):>12}")

print()
print("=" * 78)
print("KHOP HANDEDNESS (MediaPipe vs leading_hand cua HaGRID)")
print("=" * 78)
print(f"   khop       : {khop_handed['khop']}")
print(f"   lech       : {khop_handed['lech']}")
print(f"   thieu nhan : {khop_handed['thieu_nhan']}")
print(f"   ty le khop : {ty_le_khop:.1%}" if ty_le_khop is not None else "   ty le khop : N/A")

print()
print("=" * 78)
if canh_bao or (ty_le_khop is not None and ty_le_khop < 0.80):
    print("*** DUNG LAI — CHUA DUOC DUA SO NAO VAO BAO CAO ***")
    for k, t in canh_bao:
        print(f"    lop '{k}' chi phat hien {t:.1%} — duoi nguong 50%")
    if ty_le_khop is not None and ty_le_khop < 0.80:
        print(f"    ty le khop handedness {ty_le_khop:.1%} — duoi nguong 80%")
    print()
    print("    Kiem tra theo thu tu:")
    print("      1. BBOX_FORMAT co dung voi output Cell 1 khong")
    print("      2. FLIP_TRUOC_MEDIAPIPE co dang True khong")
    print("      3. Thu tang BBOX_EXPAND len 0.30-0.40 (tay bi cat mat dau ngon)")
else:
    print("OK — moi lop deu tren 50% va handedness khop tren 80%.")
    print("Tai 3 file tu /kaggle/working ve vision/model/ roi chay eval_external.py")
print("=" * 78)

---

## Sau khi chạy xong

Tải ba file từ `/kaggle/working/` về thư mục `vision/model/` trên máy:

- `keypoint_hagrid.csv`
- `keypoint_hagrid_meta.csv`
- `extract_report.json`

Rồi chạy:

```powershell
cd D:\IoT_Gesture_sequence_door_lock
.\.conda\python.exe vision\eval_external.py
```

### Ghi nguồn dữ liệu vào báo cáo

> HaGRID — HAnd Gesture Recognition Image Dataset.
> Kapitanov, Kvanchiani, Nagaev, Kraynov, Makhliarchuk.
> Giấy phép: biến thể **Creative Commons BY-SA 4.0**.
> Kho chính thức: https://github.com/hukenovs/hagrid

Phải ghi cả tên dataset lẫn giấy phép trong README và báo cáo.